# Assignment 1: Highest Grossing Films

In this work I parsed a Wikipedia page, saved the data in SQLite, and exported the data to JSON for the web page.


## Import libraries

First I import the libraries that I need for web scraping, tables, and database work.


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re

## Get the web page

Here I open the Wikipedia page and create a BeautifulSoup object.


In [13]:
# page link
url = "https://en.wikipedia.org/wiki/List_of_highest-grossing_films"

# simple header for request
headers = {"User-Agent": "Mozilla/5.0"}

# get html
response = requests.get(url, headers=headers)

# make soup object
soup = BeautifulSoup(response.text, "html.parser")

print("Loaded page:", soup.title.text)

Loaded page: List of highest-grossing films - Wikipedia


## Find the main table

This page has many tables, so I take the main wikitable and read its rows.


In [14]:
# find main table
table = soup.find("table", class_="wikitable")

# get all rows
rows = table.find_all("tr")

print("Number of rows:", len(rows))

Number of rows: 51


## Read headers

I check the headers first. This helps me find the correct columns.


In [15]:
# read header row
header_cells = rows[0].find_all(["th", "td"])
headers_list = [h.get_text(strip=True) for h in header_cells]

print("Headers:", headers_list)

# small helper to find column index
def find_col(name):
    for i, h in enumerate(headers_list):
        if name.lower() in h.lower():
            return i
    return None

title_idx = find_col("title")
gross_idx = find_col("gross")
year_idx = find_col("year")

print("Indexes:", title_idx, gross_idx, year_idx)

Headers: ['Rank', 'Peak', 'Title', 'Worldwide gross', 'Year', 'Ref']
Indexes: 2 3 4


## Clean the text

The Wikipedia table has small extra marks, so I clean the text a little.


In [16]:
# clean simple text
def clean_text(text):
    if not text:
        return None
    text = re.sub(r"\[[^\]]*\]", "", text)   # remove [1], [2]
    text = text.replace("†", "")
    text = text.replace("\xa0", " ")
    return text.strip()

# take money only
def extract_money(text):
    if not text:
        return None
    match = re.search(r"\$[\d,]+", text)
    return match.group() if match else None

# take year only
def extract_year(text):
    if not text:
        return None
    match = re.search(r"\d{4}", text)
    return int(match.group()) if match else None

## Parse film data

Now I go through the rows and save the values I need.


In [17]:
films = []

for row in rows[1:]:
    # some rows use td and some can use th
    cols = row.find_all(["td", "th"])

    if len(cols) < 5:
        continue

    title = clean_text(cols[title_idx].get_text(" ", strip=True))
    gross = extract_money(cols[gross_idx].get_text(" ", strip=True))
    year = extract_year(cols[year_idx].get_text(" ", strip=True))

    if title and gross:
        films.append({
            "title": title,
            "release_year": year,
            "director": "Unknown",
            "box_office": gross,
            "country": "Unknown"
        })

print("Films parsed:", len(films))

Films parsed: 50


## Make a DataFrame

I convert the list to a pandas DataFrame to check the result easily.


In [18]:
df = pd.DataFrame(films)

print("Preview:")
df.head(10)

Preview:


,title,release_year,director,box_office,country
0,Avatar,2009,Unknown,"$2,923,710,708",Unknown
1,Avengers: Endgame,2019,Unknown,"$2,797,501,328",Unknown
2,Avatar: The Way of Water,2022,Unknown,"$2,334,484,620",Unknown
3,Titanic,1997,Unknown,"$2,257,906,828",Unknown
4,Ne Zha 2,2025,Unknown,"$2,215,690,000",Unknown
5,Star Wars: The Force Awakens,2015,Unknown,"$2,068,223,624",Unknown
6,Avengers: Infinity War,2018,Unknown,"$2,048,359,754",Unknown
7,Spider-Man: No Way Home,2021,Unknown,"$1,922,598,800",Unknown
8,Zootopia 2,2025,Unknown,"$1,866,641,191",Unknown
9,Inside Out 2,2024,Unknown,"$1,698,863,816",Unknown


## Save data to SQLite

Here I create a table called `films` and insert the cleaned rows.


In [19]:
# connect to database
conn = sqlite3.connect("films.db")
cursor = conn.cursor()

# create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS films (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    release_year INTEGER,
    director TEXT,
    box_office TEXT,
    country TEXT
)
""")

# clear old data
cursor.execute("DELETE FROM films")

# insert rows
for _, row in df.iterrows():
    cursor.execute("""
    INSERT INTO films (title, release_year, director, box_office, country)
    VALUES (?, ?, ?, ?, ?)
    """, (
        row["title"],
        row["release_year"],
        row["director"],
        row["box_office"],
        row["country"]
    ))

conn.commit()

# quick check
check_df = pd.read_sql_query("SELECT * FROM films LIMIT 10", conn)
check_df

,id,title,release_year,director,box_office,country
0,51,Avatar,2009,Unknown,"$2,923,710,708",Unknown
1,52,Avengers: Endgame,2019,Unknown,"$2,797,501,328",Unknown
2,53,Avatar: The Way of Water,2022,Unknown,"$2,334,484,620",Unknown
3,54,Titanic,1997,Unknown,"$2,257,906,828",Unknown
4,55,Ne Zha 2,2025,Unknown,"$2,215,690,000",Unknown
5,56,Star Wars: The Force Awakens,2015,Unknown,"$2,068,223,624",Unknown
6,57,Avengers: Infinity War,2018,Unknown,"$2,048,359,754",Unknown
7,58,Spider-Man: No Way Home,2021,Unknown,"$1,922,598,800",Unknown
8,59,Zootopia 2,2025,Unknown,"$1,866,641,191",Unknown
9,60,Inside Out 2,2024,Unknown,"$1,698,863,816",Unknown


## Export to JSON

GitHub Pages is static, so I export the data to JSON and use it in JavaScript.


In [20]:
df.to_json("films.json", orient="records", indent=4, force_ascii=False)

conn.close()

print("Done")
print("Files created: films.db and films.json")

Done
Files created: films.db and films.json


## Notes

The main Wikipedia table does not directly show director and country for each film. Because of that, I used `"Unknown"` for these two fields.
